In [0]:
%pip install da


tabricks-ai-search sentence-transformers --quiet
dbutils.library.restartPython()

In [0]:
from databricks.ai_search.client import AISearchClient
from sentence_transformers import SentenceTransformer
import mlflow.deployments

CATALOG = "employee_management"
SCHEMA = "fullstack"
ENDPOINT_NAME = "employee_ai_endpoint"

vsc = AISearchClient()
client = mlflow.deployments.get_deploy_client("databricks")

general_index = vsc.get_index(ENDPOINT_NAME, f"{CATALOG}.{SCHEMA}.general_docs_index")
restricted_index = vsc.get_index(ENDPOINT_NAME, f"{CATALOG}.{SCHEMA}.restricted_docs_index")

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Ready: indexes connected, embedding model loaded, LLM client set up.")

In [0]:
def ask_ai(question, index, model_endpoint="databricks-claude-sonnet-5"):
    query_vector = model.encode([question]).tolist()[0]
    results = index.similarity_search(
        query_vector=query_vector,
        columns=["source_file", "content"],
        num_results=4
    )
    chunks = results["result"]["data_array"]
    context = "\n\n".join([f"[Source: {c[0]}]\n{c[1]}" for c in chunks])

    prompt = f"""You are an HR assistant for TechNova Solutions. Answer the employee's question using ONLY the context below. If the answer isn't in the context, say you don't have that information — do not make anything up. Keep the answer concise and cite the source document by name.

Context:
{context}

Question: {question}

Answer:"""

    response = client.predict(
        endpoint=model_endpoint,
        inputs={
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 300
        }
    )
    return response["choices"][0]["message"]["content"]

print("ask_ai function ready.")

In [0]:
answer = ask_ai("How many casual leaves do I get?", general_index, model_endpoint="databricks-meta-llama-3-1-8b-instruct")
print(answer)

In [0]:
query_vector = model.encode(["promotion"]).tolist()[0]
results = restricted_index.similarity_search(
    query_vector=query_vector,
    columns=["source_file", "content"],
    num_results=10
)
for r in results["result"]["data_array"]:
    print(r[0])